In [ ]:
import pandas as pd
import numpy as np
import torch
import transformers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModel, BertTokenizerFast, DistilBertTokenizerFast
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
import torch.nn as nn
from transformers import AdamW
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm
import os
import shap
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Read Gendered Data

In [ ]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
gendered1 = pd.read_csv('../data/letters_2021_processed_with_gender.csv')
gendered1 = gendered1[['s1_s2', 'LETTER_GENDER']]
gendered1 = gendered1.rename(columns={'LETTER_GENDER':'label'})

In [ ]:
gendered2 = pd.read_csv('../data/sentence_sets_trimmed_processed_with_gender.csv', encoding='mac-roman')
gendered2 = gendered2[['s1_s2', 'applicant_gender']]
gendered2 = gendered2.rename(columns={'TEXT':'LETTERTEXT', 'applicant_gender':'label'})

In [ ]:
gendered = pd.concat([gendered1, gendered2], ignore_index=True)

In [ ]:
gender_label_mapping = {
    'F':0,
    'female':0,
    'M':1,
    'male':1
}

In [ ]:
gendered['label'] = gendered['label'].replace(gender_label_mapping)

<ipython-input-10-d6a02b119c3b>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gendered['label'] = gendered['label'].replace(gender_label_mapping)


In [ ]:
gendered

,s1_s2,label
0,it is my pleasure to write a letter of recomme...,0
1,i am pleased to highly recommend identifier fo...,0
2,i am writing this letter in support of identif...,0
3,identifier identifier recently completed an an...,0
4,it is my pleasure to recommend dr. identifier ...,0
...,...,...
8982,it is with pleasure that i recommend first_nam...,0
8983,we are very pleased to write this letter based...,0
8984,| am writing this letter of recommendation for...,1
8985,it is my pleasure to write first_name support ...,1


In [ ]:
documents = gendered.groupby('label')['s1_s2'].apply(lambda x: ' '.join(x)).reset_index()

# Step 2: Perform TF-IDF on these grouped documents
vectorizer = TfidfVectorizer(sublinear_tf=True)
tfidf_matrix = vectorizer.fit_transform(documents['s1_s2'])

# Step 3: Get the feature names (terms)
terms = vectorizer.get_feature_names_out()

# Step 4: Convert the TF-IDF matrix to a dense array and view the results
dense_matrix = tfidf_matrix.todense()

# Step 5: Display the results in a DataFrame
tfidf_df = pd.DataFrame(dense_matrix, columns=terms, index=["Female", "Male"])
print(tfidf_df)

              00       000      0002        01     01cs        02        03  \
Female  0.006815  0.010160  0.000000  0.007291  0.00367  0.006815  0.007291   
Male    0.006459  0.009834  0.002948  0.006178  0.00000  0.005473  0.005473   

            0383        04        05  ...    zoster    zotigh  zraduates  \
Female  0.000000  0.007291  0.006815  ...  0.002612  0.000000   0.000000   
Male    0.002948  0.005473  0.005473  ...  0.002097  0.007034   0.002948   

            zsfg    zucker  zuckerberg    zumba        àö        àû        äö  
Female  0.004422  0.000000    0.004422  0.00367  0.009684  0.006815  0.021261  
Male    0.004401  0.006186    0.005473  0.00000  0.009726  0.007477  0.018889  

[2 rows x 25847 columns]


In [ ]:
# Step 6: Sort the terms by TF-IDF scores for each document
sorted_tf_idf = {}
for label in tfidf_df.index:
    sorted_tf_idf[label] = tfidf_df.loc[label].sort_values(ascending=False)

# Step 7: Display the top terms for each document
print("Top TF-IDF terms for each document:")
for label, sorted_terms in sorted_tf_idf.items():
    print(f"\nTop terms for {label} document:")
    print(sorted_terms.head(10))  # Display top 10 terms (you can adjust the number)

Top TF-IDF terms for each document:

Top terms for Female document:
and           0.030303
her           0.029729
to            0.029637
the           0.029591
she           0.029376
of            0.028978
in            0.028976
identifier    0.028113
with          0.027608
is            0.027275
Name: Female, dtype: float64

Top terms for Male document:
and           0.025898
to            0.025394
the           0.025308
he            0.025095
his           0.024897
of            0.024823
in            0.024812
identifier    0.024057
with          0.023740
is            0.023456
Name: Male, dtype: float64


In [ ]:

# Step 6: Calculate the difference between the TF-IDF scores for each term
# We subtract the Female's TF-IDF score from the Male's TF-IDF score
difference_df = tfidf_df.loc["Male"] - tfidf_df.loc["Female"]

# Step 7: Identify the terms that are unique (or important) to each gender
# We'll sort the difference by the absolute value of the differences to find the most unique terms
unique_terms_male = difference_df[difference_df > 0].sort_values(ascending=False)  # Male unique terms
unique_terms_female = difference_df[difference_df < 0].sort_values(ascending=True)  # Female unique terms

# Step 8: Display the top unique terms for each gender
print("Top unique terms for Male:")
print(unique_terms_male.head(50))  # Top 10 unique terms for Male (you can adjust the number)

print("\nTop unique terms for Female:")
print(unique_terms_female.head(50))  # Top 10 unique terms for Female (you can adjust the number)

Top unique terms for Male:
squadron         0.013670
guy              0.013428
scout            0.013164
eagle            0.012663
dylan            0.012552
2d               0.012552
jin              0.012316
himself          0.012262
lcdr             0.012059
yong             0.011778
undersea         0.011778
gas              0.011627
mr               0.011198
he               0.011139
huntington       0.011120
reese            0.011120
ordering         0.011120
aviation         0.011120
sean             0.010930
bosh             0.010930
feliciano        0.010930
saving           0.010930
roswell          0.010930
outdoor          0.010930
chung            0.010930
helicopter       0.010727
allegheny        0.010508
diagnostician    0.010508
smhs             0.010508
guitar           0.010508
suggestion       0.010508
lacking          0.010508
submarine        0.010508
rowan            0.010508
deployment       0.010272
agreeable        0.010272
removal          0.010272
brad       